# 02 - Clean Weather (Silver)

`bronze.weather` -> `silver.weather`

1. Trim text columns
2. Collapse duplicate province + year + quarter rows by averaging
   (for example South Cotabato has two weather rows per quarter). This keeps the later join one-to-one.
3. Round numbers (average first, round last)
4. Write `silver.weather`

## 1. Configuration

In [0]:
from pyspark.sql import functions as F

from common.delta_io import write_delta_table
from configs.etl_config import BRONZE_WEATHER_TABLE, SILVER_WEATHER_TABLE


print(f"Source: {BRONZE_WEATHER_TABLE}")
print(f"Target: {SILVER_WEATHER_TABLE}")

## 2. Load Bronze and trim text

In [0]:
raw_df = spark.table(BRONZE_WEATHER_TABLE)
bronze_row_count = raw_df.count()

trimmed_df = (
    raw_df
    .withColumn("province", F.trim(F.col("province")))
    .withColumn("quarter", F.trim(F.col("quarter")))
    .withColumn("year_quarter", F.trim(F.col("year_quarter")))
    .withColumn("enso_phase", F.trim(F.col("enso_phase")))
)

## 3. Collapse duplicate keys, then round

In [0]:
columns_to_average = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "precipitation_sum", "rain_sum", "precipitation_hours",
    "sunshine_duration", "shortwave_radiation_sum", "reference_evapotranspiration_mm",
    "wind_speed_10m_max", "wind_gusts_10m_max",
    "rain_normal", "temp_normal", "rainfall_deviation_pct",
    "temperature_anomaly_c", "oni_index",
]
columns_to_keep_first = ["year_quarter", "enso_phase", "_ingested_at", "_source_file"]

aggregations = [F.avg(column).alias(column) for column in columns_to_average]
aggregations += [F.first(column, ignorenulls=True).alias(column) for column in columns_to_keep_first]

averaged_df = trimmed_df.groupBy("province", "year", "quarter").agg(*aggregations)

rounding_rules = {
    0: ["sunshine_duration"],
    2: [
        "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
        "precipitation_sum", "rain_sum", "shortwave_radiation_sum",
        "reference_evapotranspiration_mm", "wind_speed_10m_max", "wind_gusts_10m_max",
        "rain_normal", "temp_normal", "rainfall_deviation_pct",
    ],
    3: ["temperature_anomaly_c", "oni_index"],
}

weather_df = averaged_df
for decimals, columns in rounding_rules.items():
    for column in columns:
        weather_df = weather_df.withColumn(column, F.round(F.col(column), decimals))

weather_df = weather_df.withColumn("precipitation_hours", F.round(F.col("precipitation_hours")).cast("int"))

## 4. Write the Silver table

In [0]:
write_delta_table(weather_df, SILVER_WEATHER_TABLE)

silver_row_count = spark.table(SILVER_WEATHER_TABLE).count()
print(f"Rows before: {bronze_row_count:,} | after: {silver_row_count:,} | duplicate keys collapsed: {bronze_row_count - silver_row_count:,}")
print("Next: 03_enrich_crop_weather")